# Variant A on Kaggle (baseline EfficientNet-B0)

Rebuilt after a Colab GPU-quota disconnect lost the previous run. Two real fixes since then:
1. **Preprocessed images are now cached** (crop + CLAHE only happens once per photo, not every epoch) - should meaningfully speed up training.
2. **Runs via Kaggle's "Save & Run All (Commit)"**, not interactive cell-by-cell execution - this runs as a background job on Kaggle's servers and saves outputs when done, independent of keeping the browser tab open. Much safer than what happened on Colab.

**Before running - set up in the Kaggle UI (not code):**
1. Right sidebar > Add Input > search and add: `mariaherrerot/ddrdataset`
2. Right sidebar > Settings > Accelerator > GPU (T4 x2 or P100)
3. Right sidebar > Settings > Internet > On (needed to clone the repo)
4. Then: **Save Version > Save & Run All (Commit)** - do not just click individual cells interactively.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/Dilshan-Fernando-01/Computer-Vision-Assignment.git
%cd Computer-Vision-Assignment

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os

# Point our code at the Kaggle-mounted dataset instead of downloading it again -
# it's already attached as an input, no need to re-fetch 3GB from Kaggle while
# already running inside Kaggle.
os.environ["DDR_GRADING_CSV"] = "/kaggle/input/ddrdataset/DR_grading.csv"
os.environ["DDR_IMAGES_DIR"] = "/kaggle/input/ddrdataset/DR_grading/DR_grading"

assert os.path.exists(os.environ["DDR_GRADING_CSV"]), "Dataset not attached - add mariaherrerot/ddrdataset via Add Input first"
print("Dataset found at", os.environ["DDR_IMAGES_DIR"])

In [ ]:
# Splits are already committed to the repo (data/processed/splits/*.csv) -
# same exact train/val/test split as the original run, no need to regenerate.
!ls data/processed/splits/

In [ ]:
import sys
sys.path.insert(0, '.')

import json
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, WeightedRandomSampler

from src.datasets.dataset import DRGradingDataset
from src.augmentation.augment import get_training_augmentations, make_sample_weights
from src.models.build_model import build_model
from src.training.train import fit, evaluate, get_device

DEVICE = get_device()
IMAGE_SIZE = 512
BATCH_SIZE = 32
EPOCHS = 40
PATIENCE = 8
LR = 1e-4

print('device:', DEVICE)

train_ds = DRGradingDataset('data/processed/splits/train.csv', image_size=IMAGE_SIZE, transform=get_training_augmentations(IMAGE_SIZE))
val_ds   = DRGradingDataset('data/processed/splits/val.csv',   image_size=IMAGE_SIZE)
test_ds  = DRGradingDataset('data/processed/splits/test.csv',  image_size=IMAGE_SIZE)

train_labels = [label for _, label in train_ds.samples]
sample_weights = make_sample_weights(train_labels)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, num_workers=4)

print('train/val/test sizes:', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
model_a = build_model('efficientnet_b0', num_classes=5, pretrained=True)

os.makedirs('outputs/checkpoints', exist_ok=True)
os.makedirs('outputs/history', exist_ok=True)

history_a = fit(
    model_a, train_loader, val_loader, criterion,
    epochs=EPOCHS, lr=LR, patience=PATIENCE,
    checkpoint_path='outputs/checkpoints/variant_a_efficientnet_b0.pt',
    device=DEVICE,
)

with open('outputs/history/variant_a_history.json', 'w') as f:
    json.dump(history_a, f, indent=2)
print('best val macro F1:', history_a['best_val_macro_f1'])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history_a['train_loss'], label='train'); axes[0].plot(history_a['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history_a['train_acc'], label='train'); axes[1].plot(history_a['val_acc'], label='val')
axes[1].set_title('Accuracy'); axes[1].legend()
axes[2].plot(history_a['val_macro_f1'], color='green'); axes[2].set_title('Val Macro F1')
fig.suptitle('Variant A - DDR (Kaggle rerun)')
fig.tight_layout()
fig.savefig('outputs/history/variant_a_curves.png', dpi=120)
plt.show()

In [ ]:
model_a.load_state_dict(torch.load('outputs/checkpoints/variant_a_efficientnet_b0.pt', map_location=DEVICE))
test_metrics_a = evaluate(model_a, test_loader, criterion, DEVICE)

report = classification_report(test_metrics_a['labels'], test_metrics_a['preds'], target_names=[f'Stage {i}' for i in range(5)])
cm = confusion_matrix(test_metrics_a['labels'], test_metrics_a['preds'])

print('Test accuracy:', test_metrics_a['accuracy'])
print('Test macro F1:', test_metrics_a['macro_f1'])
print(report)
print(np.array(cm))

with open('outputs/history/variant_a_test_report.txt', 'w') as f:
    f.write(f"Test accuracy: {test_metrics_a['accuracy']}\nTest macro F1: {test_metrics_a['macro_f1']}\n\n")
    f.write(report)
    f.write('\nConfusion matrix:\n')
    f.write(np.array2string(cm))

print('\nDone. When this notebook finishes committing, the outputs/ folder contents')
print('(checkpoint, history, curves, test report) will be saved as this version\'s Output tab -')
print('downloadable even if you closed the browser while it was running.')